# Data Understanding: Road Accident Prediction

สมุดบันทึกนี้ใช้ข้อมูลที่ผ่านการ preprocess แล้วจากโปรเจกต์เดียวกัน เพื่อสำรวจโครงสร้างข้อมูล การกระจายของข้อมูล และจุดที่ควรนำไปใช้ต่อในงานทำความสะอาดหรือสร้างโมเดล

In [3]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise FileNotFoundError("ไม่พบโฟลเดอร์ src กรุณาเปิด notebook จาก project root หรือโฟลเดอร์ notebooks")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("bmh")
plt.rcParams["figure.figsize"] = (10, 4)

from src.utils.config import PROCESSED_CSV, PROCESSED_PARQUET


In [ ]:
data_path = PROCESSED_PARQUET if PROCESSED_PARQUET.exists() else PROCESSED_CSV
if not data_path.exists():
    raise FileNotFoundError(f"ไม่พบไฟล์ข้อมูลที่ clean แล้ว: {data_path}")

if data_path.suffix == ".parquet":
    df = pd.read_parquet(data_path)
else:
    df = pd.read_csv(data_path, low_memory=False)
print(f"loaded: {data_path}")
print(f"shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()


In [ ]:
profile = (
    pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing_count": df.isna().sum(),
            "missing_pct": (df.isna().mean() * 100).round(2),
        }
    )
    .sort_values(["missing_count", "dtype"], ascending=[False, True])
)

profile.head(15)


In [ ]:
numeric_cols = [
    c
    for c in ["รวมจำนวนผู้บาดเจ็บ", "LATITUDE", "LONGITUDE", "hour", "day_of_week", "month"]
    if c in df.columns
]
df[numeric_cols].describe().T


In [ ]:
top_provinces = df["จังหวัด"].value_counts().head(10).sort_values()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].barh(top_provinces.index, top_provinces.values, color="#2a6f97")
axes[0].set_title("Top 10 จังหวัดที่เกิดอุบัติเหตุสูงสุด")
axes[0].set_xlabel("จำนวนรายการ")

axes[1].hist(df["รวมจำนวนผู้บาดเจ็บ"].dropna(), bins=30, color="#ef8354", edgecolor="white", alpha=0.9)
axes[1].set_title("Distribution ของจำนวนผู้บาดเจ็บ")
axes[1].set_xlabel("รวมจำนวนผู้บาดเจ็บ")

plt.tight_layout()
plt.show()


In [ ]:
injury_by_weather = (
    df.groupby("สภาพอากาศ")["รวมจำนวนผู้บาดเจ็บ"]
    .agg(["count", "mean", "median"])
    .sort_values("count", ascending=False)
    .head(10)
)

injury_by_weather


## สิ่งที่ควรตีความต่อ

- คอลัมน์ที่ยังมี missing สูงควรถูกติดตามต่อในขั้นตอน cleaning
- จังหวัดหรือสภาพอากาศที่มีจำนวนเหตุสูง อาจใช้เป็นหมวดสำคัญสำหรับการสร้างฟีเจอร์
- หากต้องการสร้างข้อมูลใหม่ ให้รัน `python -m src.data.run_preprocess` แล้วกลับมาเปิด notebook นี้อีกครั้ง